In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
import hashlib

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 3, Finished, Available, Finished, False)

In [2]:
df = spark.sql("SELECT * FROM MSGAnalytics.dbo.proposal_slide_content_raw")
display(df)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7834d4f2-7793-468a-aa53-0033280f756f)

## Create a clean slide-level table

In [3]:
def clean_text(text):
    if text is None:
        return None

    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = text.strip()

    return text

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 5, Finished, Available, Finished, False)

In [4]:
clean_text_udf = udf(clean_text, StringType())

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 6, Finished, Available, Finished, False)

In [5]:
cleaned_df = (
    spark.table("proposal_slide_content_raw")
    .withColumn("cleaned_text", clean_text_udf(col("combined_text")))
    .withColumn("word_count", size(split(col("cleaned_text"), " ")))
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 7, Finished, Available, Finished, False)

In [6]:
display(cleaned_df)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ccdae6eb-a500-47f4-92e0-9f22e7cca031)

### Add slide title to clean data

In [7]:
def detect_slide_title(text):
    if text is None:
        return None

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    if len(lines) == 0:
        return None

    return lines[0][:250]

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 9, Finished, Available, Finished, False)

In [8]:
detect_slide_title_udf = udf(detect_slide_title, StringType())

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 10, Finished, Available, Finished, False)

In [9]:
cleaned_df = cleaned_df.withColumn(
    "slide_title",
    detect_slide_title_udf(col("combined_text"))
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 11, Finished, Available, Finished, False)

In [10]:
display(cleaned_df)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3bf9db60-fda0-42e8-ac35-22a47ac0e609)

## Classify each slide into proposal sections

In [11]:
import json
import re
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

def has_table_data(tables_json):
    if tables_json is None:
        return False
    try:
        tables = json.loads(tables_json)
        return len(tables) > 0
    except:
        return False


def contains_dollar_amount(text):
    if text is None:
        return False

    # Matches $, $18,500, ($15,000), $ 1,000.00, etc.
    return bool(re.search(r"\(?\$\s?[\d,]+(?:\.\d{2})?\)?", text))


def detect_scope_language(text):
    if text is None:
        return False

    t = text.lower()

    scope_terms = [
        "scope",
        "scope of work",
        "proposed scope",
        "proposed work",
        "summary",
        "activities",
        "deliverables",
        "workstream"
    ]

    return any(term in t for term in scope_terms)


def classify_slide_type(cleaned_text, table_text, tables_json):
    combined = " ".join([x for x in [cleaned_text, table_text] if x])
    has_tables = has_table_data(tables_json)

    if contains_dollar_amount(combined):
        return "price_slide"

    if detect_scope_language(combined):
        return "scope_slide"

    if has_tables:
        return "structured_content_slide"

    return "narrative"

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 13, Finished, Available, Finished, False)

In [12]:
classify_slide_type_udf = udf(classify_slide_type, StringType())

typed_df = cleaned_df.withColumn(
    "slide_type",
    classify_slide_type_udf(
        col("cleaned_text"),
        col("table_text"),
        col("tables_json")
    )
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 14, Finished, Available, Finished, False)

In [13]:
display(
    typed_df
    .groupBy("slide_type")
    .count()
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 33861b5c-a5f5-410e-b0d8-dd407c9fe34b)

In [14]:
display(typed_df)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 75efc89f-6ee3-4d6f-a536-e012fc94fdea)

In [15]:
scope_slides_df = typed_df.filter(col("slide_type") == "scope_slide")
price_slides_df = typed_df.filter(col("slide_type") == "price_slide")
narrative_slides_df = typed_df.filter(col("slide_type") == "narrative")

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 17, Finished, Available, Finished, False)

## Prepare Data for Foundry

In [16]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
import re

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 18, Finished, Available, Finished, False)

In [17]:
def safe_text(value):
    if value is None:
        return ""
    return str(value).strip()


def normalize_text(text):
    if text is None:
        return None

    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def infer_agent_task(slide_type, cleaned_text):
    text = safe_text(cleaned_text).lower()

    if slide_type == "price_slide":
        return "pricing_recommendation"

    if slide_type == "scope_slide":
        return "scope_drafting"

    if any(x in text for x in ["engagement background", "why", "value", "use case", "assumptions", "next steps"]):
        return "narrative_generation"

    return "narrative_generation"


def infer_section_type(cleaned_text):
    text = safe_text(cleaned_text).lower()

    rules = [
        ("engagement_background", ["engagement background", "background"]),
        ("value_messaging", ["driving growth", "value", "business outcomes"]),
        ("use_cases", ["use case", "use cases"]),
        ("architecture", ["architecture", "future state", "lakehouse", "fabric"]),
        ("assumptions", ["assumptions"]),
        ("next_steps", ["next steps"]),
        ("scope", ["scope", "proposed work", "proposed scope"]),
        ("pricing", ["total estimated cost", "cost to client", "$"]),
    ]

    for section, keywords in rules:
        if any(k in text for k in keywords):
            return section

    return "general"

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 19, Finished, Available, Finished, False)

In [18]:
normalize_text_udf = udf(normalize_text, StringType())
infer_agent_task_udf = udf(infer_agent_task, StringType())
infer_section_type_udf = udf(infer_section_type, StringType())

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 20, Finished, Available, Finished, False)

In [19]:
base_df = (
    typed_df
    .withColumn("rag_text", normalize_text_udf(col("cleaned_text")))
    .withColumn("agent_task", infer_agent_task_udf(col("slide_type"), col("cleaned_text")))
    .withColumn("section_type", infer_section_type_udf(col("cleaned_text")))
    .withColumn("word_count", size(split(col("rag_text"), " ")))
    .filter(col("rag_text").isNotNull())
    .filter(col("word_count") >= 10)
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 21, Finished, Available, Finished, False)

In [20]:
narrative_rag_df = (
    base_df
    .filter(col("slide_type") == "narrative")
    .withColumn("content_type", lit("narrative"))
    .withColumn("title", coalesce(col("slide_title"), concat(lit("Slide "), col("slide_number"))))
    .withColumn("content", col("rag_text"))
    .withColumn(
        "metadata_json",
        to_json(struct(
            col("file_id"),
            col("customer_name"),
            col("file_path"),
            col("slide_number"),
            col("slide_title"),
            col("slide_type"),
            col("section_type"),
            col("word_count")
        ))
    )
    .select(
        sha2(concat_ws("|", col("file_id"), col("slide_number"), col("content_type")), 256).alias("knowledge_id"),
        "agent_task",
        "content_type",
        "title",
        "content",
        "metadata_json",
        col("file_path").alias("source_file_path"),
        "slide_number"
    )
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 22, Finished, Available, Finished, False)

In [21]:
scope_rag_df = (
    base_df
    .filter(col("slide_type") == "scope_slide")
    .withColumn("agent_task", lit("scope_drafting"))
    .withColumn("content_type", lit("scope"))
    .withColumn("title", coalesce(col("slide_title"), concat(lit("Scope Slide "), col("slide_number"))))
    .withColumn(
        "content",
        concat_ws(
            "\n\n",
            lit("Scope content:"),
            col("rag_text"),
            when(col("table_text").isNotNull(), concat(lit("Table content:\n"), col("table_text")))
        )
    )
    .withColumn(
        "metadata_json",
        to_json(struct(
            col("file_id"),
            col("customer_name"),
            col("file_path"),
            col("slide_number"),
            col("slide_title"),
            col("slide_type"),
            col("section_type"),
            col("word_count")
        ))
    )
    .select(
        sha2(concat_ws("|", col("file_id"), col("slide_number"), col("content_type")), 256).alias("knowledge_id"),
        "agent_task",
        "content_type",
        "title",
        "content",
        "metadata_json",
        col("file_path").alias("source_file_path"),
        "slide_number"
    )
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 23, Finished, Available, Finished, False)

In [22]:
pricing_rag_df = (
    base_df
    .filter(col("slide_type") == "price_slide")
    .withColumn("agent_task", lit("pricing_recommendation"))
    .withColumn("content_type", lit("pricing"))
    .withColumn("title", coalesce(col("slide_title"), concat(lit("Pricing Slide "), col("slide_number"))))
    .withColumn(
        "content",
        concat_ws(
            "\n\n",
            lit("Pricing content:"),
            col("rag_text"),
            when(col("table_text").isNotNull(), concat(lit("Pricing table:\n"), col("table_text")))
        )
    )
    .withColumn(
        "metadata_json",
        to_json(struct(
            col("file_id"),
            col("customer_name"),
            col("file_path"),
            col("slide_number"),
            col("slide_title"),
            col("slide_type"),
            col("section_type"),
            col("table_text"),
            col("tables_json"),
            col("word_count")
        ))
    )
    .select(
        sha2(concat_ws("|", col("file_id"), col("slide_number"), col("content_type")), 256).alias("knowledge_id"),
        "agent_task",
        "content_type",
        "title",
        "content",
        "metadata_json",
        col("file_path").alias("source_file_path"),
        "slide_number"
    )
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 24, Finished, Available, Finished, False)

In [23]:
proposal_agent_knowledge_df = (
    narrative_rag_df
    .unionByName(scope_rag_df)
    .unionByName(pricing_rag_df)
    .withColumn("created_at", current_timestamp())
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 25, Finished, Available, Finished, False)

In [24]:
proposal_agent_knowledge_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("proposal_agent_knowledge")

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 26, Finished, Available, Finished, False)

In [25]:
display(
    spark.table("proposal_agent_knowledge")
    .groupBy("agent_task", "content_type")
    .count()
    .orderBy("agent_task", "content_type")
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 199ff5ff-8580-428d-97e2-f98f93fe0093)

In [26]:
display(
    spark.table("proposal_agent_knowledge")
    .select("agent_task", "content_type", "title", "content", "metadata_json")
    .limit(20)
)

StatementMeta(, 00037096-228e-46d5-ace7-5e2b6390e6e2, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d176c922-afc8-4c28-91fb-d0c8c54b3087)